# Enterprise Data Interaction: Agentic SQLite Integration
Text search (RAG) is for vibes; SQL is for facts. When dealing with structured enterprise data, you cannot rely on vector similarity. You must give your AI the power to query your databases directly.

**The Workflow:**
1. **Schema Discovery:** The AI must first "look" at the table definitions to understand the columns.
2. **Natural Language to SQL (NL2SQL):** The AI generates a query based on the user's intent.
3. **Execution:** Your Python code runs the SQL and retrieves the raw rows.
4. **Synthesis:** The AI explains the data rows in human language.

In [1]:
import sqlite3
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

print("🗄️ SQL Agent Environment Ready!")

🗄️ SQL Agent Environment Ready!


## 1. Setting the Stage: The Inventory Database

We will create a local SQLite database representing an e-commerce inventory.

In [2]:
conn = sqlite3.connect("enterprise_inventory.db")
cursor = conn.cursor()

# Create tables
cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL,
    stock INTEGER
)
""")

# Insert Mock Data
inventory = [
    (1, 'MacBook Pro', 'Electronics', 2500.00, 15),
    (2, 'ThinkPad X1', 'Electronics', 1800.00, 20),
    (3, 'Aeron Chair', 'Furniture', 1200.00, 5),
    (4, 'Standing Desk', 'Furniture', 800.00, 10),
    (5, 'USB-C Hub', 'Accessories', 50.00, 100)
]
cursor.executemany("INSERT INTO products VALUES (?,?,?,?,?)", inventory)
conn.commit()

print("✅ Database 'enterprise_inventory.db' initialized with products table.")

✅ Database 'enterprise_inventory.db' initialized with products table.


## 2. Defining the SQL Tools

For an agent to be successful, it needs two tools: 
1. `get_db_schema`: To learn what the tables look like.
2. `run_sql_query`: To actually get the data.

In [3]:
def get_db_schema():
    """Returns the schema of the products table."""
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='products'")
    schema = cursor.fetchone()[0]
    return json.dumps({"schema": schema})

def run_sql_query(query):
    """Executes a SQL query and returns the results."""
    print(f"🛠️ [Executing SQL]: {query}")
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        return json.dumps(rows)
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_db_schema",
            "description": "Get the database schema to understand available tables and columns.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql_query",
            "description": "Execute a pure SQL query against the SQLite database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The valid SQL query to run."}
                },
                "required": ["query"]
            }
        }
    }
]

## 3. The SQL Orchestration Loop

This agent is built to handle errors. If the SQL it writes is wrong, it will receive the error message and try to fix the query in the next turn.

In [5]:
def sql_agent(user_query):
    messages = [
        {"role": "system", "content": "You are a Data Architect. Use the provided tools to answer questions about the inventory. ALWAYS check the schema before writing a query."},
        {"role": "user", "content": user_query}
    ]

    # Max 5 turns to prevent infinite loops
    for i in range(5):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS
        )
        
        msg = response.choices[0].message
        messages.append(msg)

        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                
                if func_name == "get_db_schema":
                    result = get_db_schema()
                elif func_name == "run_sql_query":
                    result = run_sql_query(args['query'])
                
                messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        else:
            # No more tool calls, return final answer
            return msg.content

print("--- Test 1: Aggregation ---")
display(Markdown(sql_agent("What is the total value of all Electronics in stock?")))

print("\n--- Test 2: Filtering ---")
display(Markdown(sql_agent("Which items have less than 10 units remaining?")))

--- Test 1: Aggregation ---
🛠️ [Executing SQL]: SELECT SUM(price * stock) AS total_value FROM products WHERE category = 'Electronics'


The total value of all Electronics in stock is $73,500.


--- Test 2: Filtering ---
🛠️ [Executing SQL]: SELECT name FROM products WHERE stock < 10;


The item with less than 10 units remaining is the "Aeron Chair".